# Day 25: Build a Vision Transformer (ViT) from Scratch

> **Goal:** Assemble everything from Days 22–24 into a complete Vision Transformer, train it on CIFAR-10, and understand ViT-specific training techniques.

---

## ViT Architecture

```
Image (3, 32, 32)
  │
  ▼
Patch Embedding + [CLS] token + Positional Encoding
  │  → (batch, N+1, d_model)
  ▼
Transformer Encoder (L layers)
  │  → (batch, N+1, d_model)
  ▼
Take [CLS] token output → (batch, d_model)
  │
  ▼
MLP Classification Head → (batch, num_classes)
```

This is remarkably simple — no convolutions, no pooling, just attention!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import time

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. ViT Building Blocks (Recap from Days 22–24)

Below we define all the components we built in the previous days, assembled in one place. If any of these feel unfamiliar, revisit the relevant day:

| Component | What it does | Built on |
|---|---|---|
| **PatchEmbedding** | Splits image into patches, adds [CLS] token + positional encoding | Day 24 |
| **MultiHeadAttention** | Computes scaled dot-product attention across multiple heads | Day 22 |
| **FeedForward** | Per-token MLP (expand → GELU → project back) | Day 23 |
| **TransformerBlock** | LayerNorm → Attention → residual → LayerNorm → FFN → residual | Day 23 |

Don't worry about memorizing every line — focus on understanding the **role** of each component in the pipeline.

In [ ]:
class PatchEmbedding(nn.Module):
    """Image → patch sequence with [CLS] token and positional encoding."""

    def __init__(self, img_size=32, patch_size=4, in_channels=3, d_model=128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_proj = nn.Conv2d(in_channels, d_model,
                                     kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model) * 0.02)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, return_weights=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ V).transpose(1, 2).reshape(B, N, C)
        out = self.proj_drop(self.proj(out))

        if return_weights:
            return out, attn
        return out


class FeedForward(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * expansion),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, expansion=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = FeedForward(d_model, expansion, dropout)

    def forward(self, x, return_weights=False):
        if return_weights:
            attn_out, weights = self.attn(self.norm1(x), return_weights=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        if return_weights:
            return x, weights
        return x

print("✅ All building blocks defined!")

---
## 2. The Complete Vision Transformer

Now we assemble everything into the full ViT. The forward pass is:

1. **Patch Embedding** — image → sequence of patch tokens + [CLS] token + positional encoding
2. **Transformer Encoder** — stack of L TransformerBlocks (self-attention + FFN)
3. **LayerNorm** — final normalization
4. **Classification Head** — take the [CLS] token → MLP → class logits

**Key detail:** We also add custom weight initialization (`_init_weights`). Transformers are sensitive to initialization — truncated normal with std=0.02 is the standard approach from the original ViT paper.

In [ ]:
class ViT(nn.Module):
    """Vision Transformer for image classification."""

    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10,
                 d_model=128, num_heads=4, num_layers=6, expansion=4, dropout=0.1):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)

        self.encoder = nn.Sequential(*[
            TransformerBlock(d_model, num_heads, expansion, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        # Classification head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        """Initialize weights for stable training."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.encoder(x)
        x = self.norm(x)
        cls_token = x[:, 0]
        logits = self.head(cls_token)
        return logits


# Create and inspect the model
model = ViT(
    img_size=32, patch_size=4, in_channels=3, num_classes=10,
    d_model=128, num_heads=4, num_layers=6, dropout=0.1
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

### Verify: Trace Data Through the ViT

Let's follow a batch of images through each stage to make sure dimensions match. This is the same debugging habit from CNNs — always trace shapes!

In [ ]:
# Verify shapes through the model
x = torch.randn(2, 3, 32, 32).to(device)
print(f"Input:         {x.shape}")

with torch.no_grad():
    patches = model.patch_embed(x)
    print(f"After embed:   {patches.shape}  (CLS + 64 patches)")

    encoded = model.encoder(patches)
    print(f"After encoder: {encoded.shape}")

    normed = model.norm(encoded)
    cls = normed[:, 0]
    print(f"CLS token:     {cls.shape}")

    logits = model.head(cls)
    print(f"Output logits: {logits.shape}")

---
## 3. Data Pipeline with Strong Augmentation

ViTs need **more augmentation** than CNNs because they lack the inductive biases (locality, translation equivariance) that CNNs have built-in.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Stronger augmentation for ViT
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
train_data, val_data = random_split(full_train, [45000, 5000], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

---
## 3. Data Pipeline with Strong Augmentation

ViTs need **more augmentation** than CNNs. Here's why:

- **CNNs have built-in inductive biases** — locality (small kernels), translation equivariance (shared weights). These help CNNs generalize even with less data.
- **ViTs have no such biases** — attention is global and content-dependent, so the model must learn spatial relationships entirely from data.
- **More augmentation compensates** by exposing the model to more variations.

We add `ColorJitter` (randomly adjust brightness, contrast, saturation) and `RandomErasing` (randomly mask out a rectangle — forces the model to not rely on any single region).

---
## 4. Training with Warmup + Cosine Decay

ViTs benefit from **learning rate warmup** — start with a very small LR and gradually increase it.

In [ ]:
def get_lr_schedule(optimizer, warmup_epochs, total_epochs, min_lr=1e-5):
    """Linear warmup + cosine decay schedule."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        else:
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return max(min_lr / optimizer.defaults["lr"],
                      0.5 * (1 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Visualize the schedule
test_opt = optim.Adam([torch.zeros(1)], lr=3e-4)
test_sched = get_lr_schedule(test_opt, warmup_epochs=5, total_epochs=50)
lrs = []
for i in range(50):
    lrs.append(test_opt.param_groups[0]["lr"])
    test_sched.step()

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch"); plt.ylabel("Learning Rate")
plt.title("Warmup + Cosine Decay Schedule")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Training and Evaluation Functions

These are the same `train_one_epoch` and `evaluate` functions we've used since Day 11 — the pattern is universal across all architectures:

In [ ]:
# Training infrastructure
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

### Train the ViT 🚀

Key ViT-specific training choices:

- **AdamW** (not plain Adam) — decouples weight decay from the adaptive learning rate, which is important for Transformers
- **Label smoothing (0.1)** — instead of hard targets (0 or 1), use soft targets (0.01 or 0.9). This prevents the model from becoming overconfident
- **Warmup + cosine decay** — start with tiny LR, ramp up, then smoothly decrease
- **Weight decay = 0.05** — stronger than typical CNN training, helps regularize the many parameters
- **50 epochs** — ViTs typically need more epochs than CNNs on small datasets

We save the best model (by validation accuracy) and restore it at the end.

In [ ]:
# Train ViT on CIFAR-10
NUM_EPOCHS = 50
WARMUP = 5

model = ViT(
    img_size=32, patch_size=4, in_channels=3, num_classes=10,
    d_model=128, num_heads=4, num_layers=6, dropout=0.1
).to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing helps ViT
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = get_lr_schedule(optimizer, WARMUP, NUM_EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
best_val_acc = 0

print(f"{'Ep':>3} | {'TrLoss':>7} | {'TrAcc':>6} | {'VLoss':>7} | {'VAcc':>6} | {'LR':>9} | {'Time':>5}")
print("-" * 65)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    tl, ta = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    vl, va = evaluate(model, val_loader, loss_fn, device)
    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()
    elapsed = time.time() - t0

    history["train_loss"].append(tl); history["val_loss"].append(vl)
    history["train_acc"].append(ta); history["val_acc"].append(va)
    history["lr"].append(lr_now)

    marker = ""
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), "best_vit.pt")
        marker = " ✓"

    if (epoch+1) % 10 == 0 or epoch == 0 or marker:
        print(f"{epoch+1:3d} | {tl:7.4f} | {ta:5.1%} | {vl:7.4f} | {va:5.1%} | {lr_now:9.6f} | {elapsed:4.1f}s{marker}")

model.load_state_dict(torch.load("best_vit.pt", map_location=device, weights_only=True))
print(f"\n✅ Best val accuracy: {best_val_acc:.2%}")

### Training Curves & Test Evaluation

The three plots below show loss, accuracy, and learning rate over training. **What to look for:**
- Loss should decrease smoothly (the warmup phase may show a brief increase at the start — that's normal)
- The gap between train and val accuracy indicates overfitting level
- The LR plot should show the warmup ramp followed by cosine decay

In [ ]:
# Plot training curves
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"], label="Val")
ax1.set_title("Loss"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], label="Train")
ax2.plot(history["val_acc"], label="Val")
ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(True, alpha=0.3)

ax3.plot(history["lr"])
ax3.set_title("Learning Rate"); ax3.grid(True, alpha=0.3)

for ax in [ax1, ax2, ax3]: ax.set_xlabel("Epoch")
plt.suptitle("ViT Training on CIFAR-10", fontsize=14)
plt.tight_layout(); plt.show()

# Test set
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
print(f"🎯 Test Accuracy: {test_acc:.2%}")

---
## 🧪 Exercises

### Exercise 1: Smaller ViT
Try d_model=64, num_layers=4, num_heads=4. How does it compare?

### Exercise 2: Without warmup
Remove the warmup phase (set warmup=0). Does training become unstable?

### Exercise 3: Patch size comparison
Train with patch_size=2 vs 4 vs 8. Compare accuracy and training speed.

### Exercise 4: Without [CLS] token
Instead of using [CLS], try averaging all patch token outputs (mean pooling). Does it work as well?

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **ViT** | Patch embed → Transformer encoder → [CLS] → MLP head |
| **No convolutions!** | Pure attention-based architecture |
| **Strong augmentation** | ViT needs more augmentation than CNNs |
| **Warmup + cosine decay** | Critical for stable ViT training |
| **Label smoothing** | Prevents overconfident predictions |
| **AdamW** | Adam with decoupled weight decay |
| **Weight init** | Truncated normal (std=0.02) for stability |

**Tomorrow:** We'll visualize what the ViT's attention heads learn — which patches attend to which!